# Fako Online - B-Roll Generation Server (Kaggle)

**Wan 2.1 (1.3B) Video Generation**

Generates short B-Roll video clips from text prompts.

### Instructions
1. Enable **GPU T4 x2** (Settings > Accelerator)
2. Add dataset `kingtechie/wan21-model`
3. Run all cells in order
4. The server will start on port 8001

In [ ]:
# Cell 1: Setup paths
import os

WORKING_DIR = "/kaggle/working/outputs"
os.makedirs(WORKING_DIR, exist_ok=True)

WAN_MODEL_DIR = "/kaggle/input/wan21-model"

print(f"Wan Model: {WAN_MODEL_DIR}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q diffusers transformers accelerate
!pip install -q fastapi uvicorn python-multipart
!pip install -q imageio[ffmpeg]
print("Dependencies installed!")

In [ ]:
# Cell 3: Import libraries
import torch
import numpy as np
from diffusers import WanPipeline
from diffusers.utils import export_to_video
import io
import base64

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Cell 4: Load Wan 2.1 pipeline
pipe = WanPipeline.from_pretrained(
    WAN_MODEL_DIR,
    torch_dtype=torch.float16
)
pipe.to("cuda")

print("Wan 2.1 loaded!")

In [ ]:
# Cell 5: B-Roll generation function
def generate_broll(prompt, duration=5, num_frames=81):
    video_frames = pipe(
        prompt=prompt,
        num_frames=num_frames,
        num_inference_steps=20,
        guidance_scale=7.5,
        height=480,
        width=832
    ).frames[0]
    output_path = f"{WORKING_DIR}/broll_clip.mp4"
    export_to_video(video_frames, output_path, fps=24)
    return output_path

print("B-Roll generation function defined!")

In [ ]:
# Cell 6: FastAPI server
from fastapi import FastAPI, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title="Fako Online - B-Roll API")

@app.get("/health")
async def health():
    return {"status": "ok", "model": "wan-2.1-1.3b"}

@app.post("/generate-broll")
async def api_generate_broll(text_prompt: str = Form(...), duration: int = Form(5)):
    try:
        video_path = generate_broll(text_prompt, duration)
        return FileResponse(video_path, media_type="video/mp4")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

print("FastAPI server defined!")

In [ ]:
# Cell 7: Start server
print("Starting server on port 8001...")
uvicorn.run(app, host="0.0.0.0", port=8001)